# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent)

**Quy ước ma trận:**
- `0` → Ô trống (sạch) / Máy hút bụi
- `1` → Bụi

**Thuật toán:** Backtracking — DFS khám phá lưới, quay lui khi gặp ngõ cụt.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
DUST_PROB = 0.4

# ── Tạo môi trường ──
def create_env(rows, cols, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            if random.random() < dust_prob:
                grid[r][c] = 1
    return grid

# ── Vẽ ma trận ──
def draw_grid(grid, pos, title):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(cols * 0.9, rows * 0.9))

    cmap = ListedColormap(['#F0F0F0', '#F4D03F'])  # 0=xám nhạt, 1=vàng
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)

    for r in range(rows):
        for c in range(cols):
            if (r, c) == pos:
                ax.text(c, r, '🤖', ha='center', va='center', fontsize=14)
            elif grid[r][c] == 1:
                ax.text(c, r, '●', ha='center', va='center',
                        fontsize=16, color='#884400')
            else:
                ax.text(c, r, '0', ha='center', va='center',
                        fontsize=11, color='#888888')

    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xticklabels(np.arange(cols))
    ax.set_yticklabels(np.arange(rows))
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which='minor', color='#AAAAAA', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)

    legend = [
        mpatches.Patch(color='#F0F0F0', label='0 - Ô sạch / Máy'),
        mpatches.Patch(color='#F4D03F', label='1 - Bụi'),
    ]
    ax.legend(handles=legend, loc='upper right',
              bbox_to_anchor=(1.35, 1.02), fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Khởi tạo ──
grid = create_env(ROWS, COLS, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu  |  Tổng bụi: {total_dust} ô")
draw_grid(grid, pos=(-1, -1), title=f'Ma trận ban đầu — Bụi: {total_dust} ô')

In [ ]:
# ── Agent - Backtracking (DFS + Quay lui) ──
def run_agent(grid_in):
    grid = grid_in.copy()
    rows, cols = grid.shape
    steps = 0
    cleaned = 0

    # Tập các ô đã thăm
    visited = set()
    path_stack = []  # Stack lưu đường đi để quay lui

    # Hướng di chuyển: phải, xuống, trái, lên
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    dir_names = {(0, 1): 'PHẢI', (1, 0): 'XUỐNG', (0, -1): 'TRÁI', (-1, 0): 'LÊN'}

    # Bắt đầu tại ô (0, 0)
    r, c = 0, 0
    visited.add((r, c))
    path_stack.append((r, c))

    print(f'Bắt đầu tại ({r}, {c})')
    if grid[r][c] == 1:
        grid[r][c] = 0
        cleaned += 1
        print(f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}')
    else:
        print(f'   ⟹  Ô sạch, tiếp tục.')
    draw_grid(grid, pos=(r, c),
              title=f'Bắt đầu | Vị trí: ({r},{c}) | Đã hút: {cleaned}/{total_dust}')

    # Tiếp tục cho đến khi tất cả các ô đã được thăm
    while len(visited) < rows * cols:
        moved = False

        # Thử các hướng theo thứ tự ưu tiên
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and (nr, nc) not in visited:
                steps += 1
                direction = dir_names[(dr, dc)]
                move_msg = f'Bước {steps}: Di chuyển {direction} → ô ({nr}, {nc})'

                r, c = nr, nc
                visited.add((r, c))
                path_stack.append((r, c))

                print(move_msg)

                if grid[r][c] == 1:
                    grid[r][c] = 0
                    cleaned += 1
                    action_msg = f'   ⟹  Phát hiện bụi! Hút bụi tại ({r}, {c}) — Đã hút: {cleaned}/{total_dust}'
                else:
                    action_msg = f'   ⟹  Ô sạch, tiếp tục.'

                print(action_msg)
                draw_grid(grid, pos=(r, c),
                          title=f'Bước {steps} | Vị trí: ({r},{c}) | Đã hút: {cleaned}/{total_dust}')

                moved = True
                break

        if not moved:
            # Backtrack: quay lui về ô trước đó trong stack
            if len(path_stack) > 1:
                path_stack.pop()  # Bỏ ô hiện tại (đã bị kẹt)
                pr, pc = path_stack[-1]  # Ô sẽ quay lui về

                steps += 1
                # Xác định hướng quay lui
                if r > pr:
                    direction = 'LÊN (quay lui)'
                elif r < pr:
                    direction = 'XUỐNG (quay lui)'
                elif c > pc:
                    direction = 'TRÁI (quay lui)'
                else:
                    direction = 'PHẢI (quay lui)'

                move_msg = f'Bước {steps}: Quay lui {direction} → ô ({pr}, {pc})'
                print(move_msg)
                print(f'   ⟹  Ngõ cụt! Không còn ô chưa thăm, quay lui về nút cha.')

                r, c = pr, pc
                draw_grid(grid, pos=(r, c),
                          title=f'Bước {steps} | Vị trí: ({r},{c}) | Quay lui | Đã hút: {cleaned}/{total_dust}')
            else:
                # Đã quay lui về điểm bắt đầu và không còn hướng nào
                break

    # Kết quả
    print('=' * 45)
    if cleaned == total_dust:
        status = 'THÀNH CÔNG'
        reason = 'Đã duyệt toàn bộ lưới bằng backtracking, hút sạch hết bụi'
    else:
        status = 'THẤT BẠI'
        reason = 'Không hút hết bụi'

    print(f'Số bước đi  : {steps}')
    print(f'Bụi đã hút  : {cleaned} / {total_dust} ô')
    print(f'Trạng thái  : {status}')
    print(f'Lý do       : {reason}')


run_agent(grid)